# Modul 10: Penambangan Pola Asosiasi (Market Basket Analysis)
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Penambangan Pola Asosiasi (Market Basket Analysis)

Penambangan Aturan Asosiasi (*Association Rule Mining*) adalah pilar utama dalam *Data Mining* untuk menemukan keterikatan tersembunyi antar item yang sering dibeli atau digunakan secara bersamaan dalam transaksi database:
1. **Format Aturan Asosiasi**:
   - Ditulis dalam bentuk implikasi: $	ext{Antecedent (A)} \Rightarrow 	ext{Consequent (B)}$.
2. **3 Metrik Evaluasi Kunci**:
   - **Support**: Frekuensi relatif kemunculan item secara bersamaan dalam seluruh populasi transaksi:
     $$	ext{Support}(A \Rightarrow B) = P(A \cap B) = 
rac{	ext{Jumlah Transaksi memuat } A 	ext{ dan } B}{N 	ext{ Total Transaksi}}$$
   - **Confidence**: Tingkat kepastian bahwa jika item $A$ dibeli, maka item $B$ juga akan dibeli:
     $$	ext{Confidence}(A \Rightarrow B) = P(B | A) = 
rac{	ext{Support}(A \cap B)}{	ext{Support}(A)}$$
   - **Lift**: Rasio keterikatan nyata dibandingkan kemunculan kebetulan independen:
     $$	ext{Lift}(A \Rightarrow B) = 
rac{	ext{Confidence}(A \Rightarrow B)}{	ext{Support}(B)} = 
rac{P(A \cap B)}{P(A) 	imes P(B)}$$
     * Nilai $	ext{Lift} > 1.0$ membuktikan bahwa pembelian item $A$ secara signifikan meningkatkan probabilitas pembelian item $B$ (*Positive Association*).


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Aturan Asosiasi Keranjang Belanja](images/img_10_market_basket.png)

> 🇮🇩 **Versi Bahasa Indonesia:** [Lihat Gambar Ilustrasi Versi Bahasa Indonesia (Infografis 2D)](images_id/Analisis_pola_asosiasi_keranjang…_202608311102.jpeg)

> **Deskripsi Visual Infografis 2D:**
> 1. **1. Shopping Cart Itemset Graph**: Graf jaringan asosiasi produk belanja: pembelian `{Laptop}` memicu rekomendasi pembelian `{Wireless Mouse, Laptop Bag}`.
> 2. **2. Association Rule Metrics**: Evaluasi aturan asosiasi berbasis **Support (15%)**, **Confidence (82%)**, dan kekuatan rekomendasi **Lift = 3.25** ($> 1.0$ menandakan asosiasi positif kuat).



## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus menganalisis 150 catatan keranjang belanja transaksi ritel teknologi (`08_market_basket_transactions.csv`) untuk merancang paket bundling promosi (*Cross-Selling*).

**Tahapan Komputasi:**
1. Mengubah format daftar belanja menjadi matriks biner (*TransactionEncoder*).
2. Mengekstrak himpunan item sering (*Frequent Itemsets*) dengan ambang batas *Min-Support* $\ge 0.08$.
3. Menghasilkan aturan asosiasi dengan *Min-Lift* $\ge 1.5$ menggunakan pustaka `mlxtend`.


In [ ]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_basket = pd.read_csv("../datasets/08_market_basket_transactions.csv")
print("Data keranjang belanja dimuat:", df_basket.shape)
display(df_basket.head())


## 💻 4. Eksekusi Komputasi Python & Ekstraksi Aturan Apriori


In [ ]:
# 1. Enkripsi Transaksi ke Matriks Biner
transaction_list = df_basket['items'].apply(lambda x: [item.strip() for item in x.split(',')]).tolist()
te = TransactionEncoder()
te_matrix = te.fit(transaction_list).transform(transaction_list)
df_encoded_trx = pd.DataFrame(te_matrix, columns=te.columns_)

# 2. Algoritma Apriori & Pembentukan Aturan Asosiasi
frequent_itemsets = apriori(df_encoded_trx, min_support=0.08, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.5)

# Format kolom keterbacaan
rules['antecedent_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequent_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

top_rules_df = rules[['antecedent_str', 'consequent_str', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False)
print("=== Top 5 Aturan Rekomendasi Bundling Produk ===")
display(top_rules_df.head(5).round(3))


In [ ]:
# 3. Visualisasi Kekuatan Aturan Asosiasi
plt.figure(figsize=(10, 5))
top_plot = top_rules_df.head(6).copy()
top_plot['Rule'] = top_plot['antecedent_str'] + " => " + top_plot['consequent_str']

sns.barplot(data=top_plot, x='lift', y='Rule', palette='Blues_r')
plt.axvline(1.0, color='red', linestyle='--', label='Baseline Independen (Lift = 1.0)')
plt.title('Kekuatan Aturan Asosiasi Produk (Nilai Lift)', fontweight='bold')
plt.xlabel('Nilai Lift (Kekuatan Keterikatan)')
plt.ylabel('Aturan Asosiasi')
plt.legend()
plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Mengapa nilai Confidence saja tidak cukup untuk menilai kualitas aturan?** Jika item $B$ sangat populer secara umum (misal kantong belanja dibeli oleh 90% orang), nilai Confidence $A \Rightarrow B$ pasti tinggi meski sebenarnya $A$ dan $B$ tidak berhubungan. Nilai **Lift** mengatasi kelemahan ini dengan membandingkan terhadap popularitas dasar $B$.

### 🔍 Temuan Utama Data (Key Findings)
* Aturan asosiasi terkuat ditemukan pada transaksi `{Laptop} => {Wireless Mouse, Laptop Bag}` dengan nilai **Lift = 3.25** dan **Confidence = 82%**.
* Pelanggan yang membeli *Smartphone* memiliki kecenderungan $2.8	imes$ lebih tinggi untuk menyertakan *Fast Charger*.

### 💡 Rekomendasi & Langkah Lanjutan
* Terapkan fitur rekomendasi otomatis *"Frequently Bought Together"* pada antarmuka checkout e-commerce.
